In [ ]:
import regex as re


In [ ]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

In [ ]:
text = """The 2026 ICC Women's T20 World Cup was the tenth edition of the ICC Women's T20 World Cup and was hosted by the England and Wales Cricket Board from 12 June to 5 July 2026. England had previously hosted the inaugural competition in 2009. A total of twelve teams competed in 33 matches across seven venues in England.

The number of participants was increased from ten teams to twelve, which included a host team, top five teams from the previous edition, the two highest-ranked teams in the ICC Women's T20I Team Rankings not already qualified, and four other teams determined through a series of qualifiers. Netherlands qualified for the Women's T20 World Cup for the first time.

New Zealand were the defending champions and were eliminated in the group stage. Australia defeated England by 7 wickets in the final to win their record seventh title. Teams that finished higher from the continents of Africa, Asia, Europe and Oceania: South Africa, India, Great Britain (taking the place of England) and Australia"""

In [ ]:
text

In [ ]:
compile_pattern = re.compile(GPT4_SPLIT_PATTERN)

In [ ]:
compile_pattern

#### Chunking by re pattern

In [ ]:
text_chunks = re.findall(compile_pattern, text)
text_chunks

#### Encoding chunks to utf-8

In [ ]:
ids = [list(ch.encode("utf-8")) for ch in text_chunks]
ids

In [ ]:
vocab_size = 276
assert vocab_size >= 256
num_merges = vocab_size - 256
num_merges

##### Initialized vocab with 256 ascii codes

In [ ]:
merges = {}
vocab = {idx: bytes([idx]) for idx in range(256)}
vocab

#### Helper func: 
#####   get_stats:- 
    Given a list of integers, return a dictionary of counts of consecutive pairs
    Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    Optionally allows to update an existing dictionary of counts

##### merge:- 
     In the list of integers (ids), replace all consecutive occurrences
    of pair with the new integer token idx
    Example: ids=[1, 2, 3, 1, 2], pair=(1, 2), idx=4 -> [4, 3, 4]

In [ ]:
def get_stats(ids, count=None):
    count = {} if count is None else count
    for pair in zip(ids, ids[1:]):
        count[pair] = count.get(pair, 0) + 1
    return count

def merge(ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


## Training loop

In [ ]:
for i in range(num_merges):
    stats = {}
    for chunk_ids in ids:
        get_stats(chunk_ids, stats)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = [merge(chunk_ids , pair, idx) for chunk_ids in ids]
    merges[pair] = idx
    vocab[idx]  = vocab[pair[0]] + vocab[pair[1]]
    print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

In [ ]:
merges

In [ ]:
vocab

### Special tokens

In [ ]:
special_token = {}
inverse_special_token = {v:k for k, v in special_token.items()}
inverse_special_token

## Decode

In [ ]:
sample_ids = [84, 257, 32, 110, 117, 109, 98, 101, 114, 274, 102, 32, 112, 97, 114, 116, 105, 99, 105, 112, 259, 116, 115, 270, 97, 115, 268, 99, 114, 263, 115, 261, 265, 114, 266, 256, 264, 256, 272, 115, 256, 111, 256, 119, 101, 108, 118, 101, 44, 270, 104, 105, 99, 104, 268, 99, 108, 117, 100, 261, 32, 97, 32, 104, 111, 271, 256, 272, 44, 256, 111, 112, 265, 105, 118, 101, 256, 272, 115, 265, 114, 266, 260, 32, 112, 114, 101, 118, 105, 111, 117, 115, 32, 261, 105, 116, 105, 111, 110, 44, 260, 256, 119, 111, 32, 104, 105, 103, 257, 271, 45, 114, 259, 107, 261, 256, 272, 115, 268, 260, 32, 73, 67, 67, 269, 266, 264, 39, 115, 32, 84, 273, 73, 32, 84, 272, 32, 82, 259, 107, 258, 103, 115, 32, 110, 111, 116, 32, 267, 114, 263, 100, 121, 32, 113, 117, 267, 105, 102, 105, 261, 44, 32, 262, 265, 111, 117, 114, 274, 116, 257, 114, 256, 272, 115, 32, 100, 101, 116, 101, 114, 109, 258, 261, 256, 104, 114, 111, 117, 103, 104, 32, 97, 32, 115, 101, 114, 105, 101, 115, 274, 102, 32, 113, 117, 267, 105, 102, 105, 101, 114, 115, 46, 32, 78, 101, 116, 257, 114, 275, 115, 32, 113, 117, 267, 105, 102, 105, 261, 265, 111, 114, 260, 269, 266, 264, 39, 115, 32, 84, 273, 269, 111, 114, 108, 100, 32, 67, 117, 112, 265, 111, 114, 260, 265, 105, 114, 271, 256, 105, 109, 101, 46, 60, 124, 264, 100, 111, 102, 116, 101, 120, 116, 124, 62]

In [ ]:
# # given ids (list of integers), return Python string

part_bytes = []
for i in sample_ids:
    if i in vocab:
        part_bytes.append(vocab[i])
    elif i in inverse_special_token:
        part_bytes.append(inverse_special_token[i].encode("utf-8"))
    else:
        raise ValueError(f'invalid token id: {i}')

text_bytes = b''.join(part_bytes)
text = text_bytes.decode("utf-8", errors="replace")
print(text)

## Encode chunk

     Helper funtions

In [ ]:
def encode_chunk( text_bytes):
    ids = list(text_bytes)
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key= lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        ids = merge(ids, pair, idx)
    return ids

def encode_ordinary(text):
    text_chunks = re.findall(compile_pattern, text)
    ids = []

    for chunk in text_chunks:
        chunk_bytes = chunk.encode("utf-8")
        chunk_ids = encode_chunk(chunk_bytes)
        ids.extend(chunk_ids)
    return ids


     Encoder 

In [ ]:
allowed_special = "all"

In [ ]:
sample_text = '''The number of participants was increased from ten teams to twelve, which included a host team, top five teams from the previous edition, the two highest-ranked teams in the ICC Women's T20I Team Rankings not already qualified, and four other teams determined through a series of qualifiers. Netherlands qualified for the Women's T20 World Cup for the first time.<|endoftext|>'''

In [ ]:
special = None
if allowed_special == "all":
    special = special_token
elif allowed_special == "none":
    special = {}
elif allowed_special == "none_raise":
    special = {}
    assert all(token not in sample_text for token in special_token)
elif isinstance(allowed_special, set):
    special = {k:v for k, v in special_token.items() if k in allowed_special}
else:
    raise ValueError(f'allowed_special={allowed_special} not understood')
if not special:
    ids = encode_ordinary(sample_text)
else:
    special_pattern = "(" + "|".join(re.escape(k) for k in special) + ")"
    special_chunks = re.split(special_pattern, sample_text)

    ids = []
    for part in special_chunks:
        if part in special:
            ids.append(special[part])
        else:
            ids.extend(encode_ordinary(part))

print(ids)

### Model usage

In [ ]:
sample_text = '''The number of participants was increased from ten teams to twelve, which included a host team, top five teams from the previous edition, the two highest-ranked teams in the ICC Women's T20I Team Rankings not already qualified, and four other teams determined through a series of qualifiers. Netherlands qualified for the Women's T20 World Cup for the first time.<|endoftext|>'''

In [ ]:
from regex_tokenizer import BPETokenizer

tokenizer = BPETokenizer()
tokenizer.load("models\\bpe.model")
result = tokenizer.encode(sample_text, allowed_special="all")

In [ ]:
print(result)

In [ ]:
text = tokenizer.decode(result)
text = text.encode("utf-8")
text

In [ ]:
print(tokenizer.vocab)

In [ ]:
tokenizer.special_tokens